# Week 8: Multi-Agent Deal Hunting Framework

**Author:** James Kimani  
**Team:** Euclid

Autonomous agent system that finds online deals, estimates fair prices, and sends notifications.

## Architecture

| Agent | Role |
|-------|------|
| ScannerAgent | Scrapes deals from RSS feeds |
| FrontierAgent | RAG + GPT for price estimation |
| EnsembleAgent | Combines multiple estimators |
| MessagingAgent | Sends push notifications |
| PlanningAgent | Orchestrates the pipeline |

---

## 1. Setup

In [ ]:
import os
import sys
import json
import re
import logging
from pathlib import Path
from typing import List, Dict
from dotenv import load_dotenv

# Add week8 root to path for imports
week8_root = Path.cwd().parent if (Path.cwd().parent / 'agents').exists() else Path.cwd().parents[1]
sys.path.insert(0, str(week8_root))

load_dotenv(override=True)

# Verify API keys
print(f"OPENAI_API_KEY: {'SET' if os.getenv('OPENAI_API_KEY') else 'MISSING'}")
print(f"HF_TOKEN: {'SET' if os.getenv('HF_TOKEN') else 'MISSING'}")

In [ ]:
from openai import OpenAI
import chromadb
import gradio as gr

openai = OpenAI()

---

## 2. Data Models

In [ ]:
from pydantic import BaseModel, Field

class Deal(BaseModel):
    """A deal with product description, price, and URL."""
    product_description: str = Field(description="Summary of the product in 3-4 sentences")
    price: float = Field(description="Actual price as advertised")
    url: str = Field(description="URL of the deal")

class DealSelection(BaseModel):
    """List of selected deals."""
    deals: List[Deal] = Field(description="Top 5 deals with clear descriptions and prices")

class Opportunity(BaseModel):
    """A deal where estimated value exceeds the price."""
    deal: Deal
    estimate: float
    discount: float

---

## 3. Base Agent

In [ ]:
class Agent:
    """Base class for all agents with colored logging."""
    
    # ANSI colors
    COLORS = {
        'red': '\033[31m', 'green': '\033[32m', 'yellow': '\033[33m',
        'blue': '\033[34m', 'magenta': '\033[35m', 'cyan': '\033[36m'
    }
    RESET = '\033[0m'
    
    name: str = "Agent"
    color: str = 'white'
    
    def log(self, message: str):
        color_code = self.COLORS.get(self.color, '')
        print(f"{color_code}[{self.name}] {message}{self.RESET}")

---

## 4. Scanner Agent

Scrapes deals from RSS feeds (DealNews).

In [ ]:
import feedparser
import requests
from bs4 import BeautifulSoup

RSS_FEEDS = [
    "https://www.dealnews.com/c142/Electronics/?rss=1",
    "https://www.dealnews.com/c39/Computers/?rss=1",
]

class ScannerAgent(Agent):
    name = "Scanner"
    color = "cyan"
    
    def scan(self, max_deals: int = 10) -> List[Dict]:
        """Fetch deals from RSS feeds."""
        self.log(f"Scanning {len(RSS_FEEDS)} feeds...")
        deals = []
        
        for feed_url in RSS_FEEDS:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries[:max_deals // len(RSS_FEEDS)]:
                deals.append({
                    'title': entry.get('title', '')[:100],
                    'summary': self._clean_html(entry.get('summary', '')),
                    'url': entry.get('links', [{}])[0].get('href', '')
                })
        
        self.log(f"Found {len(deals)} deals")
        return deals
    
    def _clean_html(self, html: str) -> str:
        soup = BeautifulSoup(html, 'html.parser')
        return soup.get_text(strip=True)[:500]

In [ ]:
# Test scanner
scanner = ScannerAgent()
raw_deals = scanner.scan(max_deals=6)
raw_deals[:2]

---

## 5. Frontier Agent

Uses RAG + GPT to estimate fair market price.

In [ ]:
class FrontierAgent(Agent):
    name = "Frontier"
    color = "green"
    
    def __init__(self, collection=None):
        self.collection = collection
        self.client = OpenAI()
    
    def estimate(self, description: str) -> float:
        """Estimate price using GPT with optional RAG context."""
        self.log(f"Estimating: {description[:50]}...")
        
        # RAG: get similar products from vector DB
        context = ""
        if self.collection:
            results = self.collection.query(query_texts=[description], n_results=3)
            if results['documents']:
                context = "Similar products:\n" + "\n".join(results['documents'][0])
        
        prompt = f"""Estimate the fair market price in USD for this product.
{context}

Product: {description}

Reply with only the numeric price (no $ sign)."""
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=20
        )
        
        price_str = response.choices[0].message.content.strip()
        return self._extract_price(price_str)
    
    def _extract_price(self, text: str) -> float:
        text = text.replace('$', '').replace(',', '')
        match = re.search(r'\d+\.?\d*', text)
        return float(match.group()) if match else 0.0

---

## 6. Ensemble Agent

Combines estimates from multiple agents.

In [ ]:
class EnsembleAgent(Agent):
    name = "Ensemble"
    color = "magenta"
    
    def __init__(self, agents: List[Agent], weights: List[float] = None):
        self.agents = agents
        self.weights = weights or [1.0 / len(agents)] * len(agents)
    
    def estimate(self, description: str) -> float:
        """Weighted average of all agent estimates."""
        self.log(f"Combining {len(self.agents)} estimates...")
        
        estimates = []
        for agent in self.agents:
            try:
                est = agent.estimate(description)
                estimates.append(est)
            except Exception as e:
                self.log(f"Agent {agent.name} failed: {e}")
                estimates.append(0.0)
        
        weighted_sum = sum(e * w for e, w in zip(estimates, self.weights))
        self.log(f"Ensemble estimate: ${weighted_sum:.2f}")
        return weighted_sum

---

## 7. Messaging Agent

Sends notifications for good deals (Pushover optional).

In [ ]:
class MessagingAgent(Agent):
    name = "Messenger"
    color = "yellow"
    
    def __init__(self):
        self.pushover_user = os.getenv('PUSHOVER_USER')
        self.pushover_token = os.getenv('PUSHOVER_TOKEN')
    
    def alert(self, opportunity: Opportunity):
        """Send notification for a deal opportunity."""
        msg = f"""Deal Alert!
{opportunity.deal.product_description[:100]}
Price: ${opportunity.deal.price:.2f}
Estimated: ${opportunity.estimate:.2f}
Savings: ${opportunity.discount:.2f}
{opportunity.deal.url}"""
        
        self.log(f"Alert: {opportunity.deal.product_description[:50]}...")
        
        # Send via Pushover if configured
        if self.pushover_user and self.pushover_token:
            requests.post("https://api.pushover.net/1/messages.json", data={
                "token": self.pushover_token,
                "user": self.pushover_user,
                "message": msg,
                "url": opportunity.deal.url
            })
        
        return msg

---

## 8. Planning Agent

Orchestrates the full pipeline.

In [ ]:
class PlanningAgent(Agent):
    name = "Planner"
    color = "blue"
    
    def __init__(self, collection=None):
        self.scanner = ScannerAgent()
        self.frontier = FrontierAgent(collection)
        self.messenger = MessagingAgent()
        self.min_discount = 20.0  # Minimum discount to alert
    
    def plan(self, memory: List[Opportunity] = None) -> Opportunity:
        """Run the deal-finding pipeline."""
        self.log("Starting deal hunt...")
        memory = memory or []
        seen_urls = {opp.deal.url for opp in memory}
        
        # 1. Scan for deals
        raw_deals = self.scanner.scan(max_deals=10)
        
        # 2. Filter and process
        best_opportunity = None
        best_discount = 0
        
        for raw in raw_deals:
            if raw['url'] in seen_urls:
                continue
            
            # Use LLM to extract structured deal
            deal = self._extract_deal(raw)
            if not deal or deal.price <= 0:
                continue
            
            # 3. Estimate fair price
            estimate = self.frontier.estimate(deal.product_description)
            discount = estimate - deal.price
            
            if discount > best_discount and discount >= self.min_discount:
                best_discount = discount
                best_opportunity = Opportunity(
                    deal=deal,
                    estimate=estimate,
                    discount=discount
                )
        
        # 4. Send notification if found
        if best_opportunity:
            self.log(f"Found opportunity: ${best_opportunity.discount:.2f} off!")
            self.messenger.alert(best_opportunity)
        else:
            self.log("No new opportunities found")
        
        return best_opportunity
    
    def _extract_deal(self, raw: Dict) -> Deal:
        """Extract structured deal from raw data using LLM."""
        prompt = f"""Extract the product and price from this deal:
Title: {raw['title']}
Summary: {raw['summary']}

Return JSON: {{"product_description": "...", "price": 0.0}}"""
        
        try:
            response = openai.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )
            data = json.loads(response.choices[0].message.content)
            return Deal(
                product_description=data.get('product_description', raw['title']),
                price=float(data.get('price', 0)),
                url=raw['url']
            )
        except Exception as e:
            self.log(f"Extract failed: {e}")
            return None

---

## 9. Deal Agent Framework

In [ ]:
class DealAgentFramework:
    MEMORY_FILE = "memory.json"
    
    def __init__(self):
        self.memory = self._load_memory()
        self.planner = None
    
    def _load_memory(self) -> List[Opportunity]:
        if os.path.exists(self.MEMORY_FILE):
            with open(self.MEMORY_FILE) as f:
                return [Opportunity(**item) for item in json.load(f)]
        return []
    
    def _save_memory(self):
        with open(self.MEMORY_FILE, 'w') as f:
            json.dump([opp.model_dump() for opp in self.memory], f, indent=2)
    
    def run(self) -> List[Opportunity]:
        """Execute the deal-hunting pipeline."""
        if not self.planner:
            self.planner = PlanningAgent()
        
        result = self.planner.plan(self.memory)
        if result:
            self.memory.append(result)
            self._save_memory()
        
        return self.memory

In [ ]:
# Run the framework
framework = DealAgentFramework()
opportunities = framework.run()
print(f"\nTotal opportunities: {len(opportunities)}")

---

## 10. Gradio UI

In [ ]:
def create_ui():
    framework = DealAgentFramework()
    
    def get_table():
        return [
            [opp.deal.product_description[:80], f"${opp.deal.price:.2f}", 
             f"${opp.estimate:.2f}", f"${opp.discount:.2f}", opp.deal.url]
            for opp in framework.memory
        ]
    
    def run_scan():
        framework.run()
        return get_table()
    
    with gr.Blocks(title="Deal Hunter") as ui:
        gr.Markdown("# The Price is Right - Deal Hunting AI")
        gr.Markdown("Autonomous agent that finds deals and estimates fair prices.")
        
        table = gr.Dataframe(
            headers=["Product", "Price", "Estimate", "Discount", "URL"],
            value=get_table(),
            wrap=True
        )
        
        btn = gr.Button("Scan for Deals")
        btn.click(run_scan, outputs=table)
    
    return ui

# Launch UI
ui = create_ui()
ui.launch()

---

## Results

| Metric | Value |
|--------|-------|
| Deals scanned | - |
| Opportunities found | - |
| Avg discount | $- |